# Text Branch — Data, Baselines, and Word-Level Explanations

[![GitHub](https://img.shields.io/badge/GitHub-manasdutta04%2Fmultimodal--fake--news--detector-181717?logo=github&logoColor=white)](https://github.com/manasdutta04/multimodal-fake-news-detector)

This notebook implements the **text-only** path of *Multimodal Fake News Detector with Explainability*: binary real/fake classification on Fakeddit titles, plus token-level attribution.

It is the first trained component of the system. Downstream notebooks reuse the official splits, the DistilBERT checkpoint, and these metrics as the unimodal text baseline for fusion.



**Scope**
- Dataset: Fakeddit multimodal-only TSVs (`clean_title`, `2_way_label`)
- Models: TF-IDF + logistic regression (baseline); DistilBERT (primary text encoder)
- Metrics: accuracy, per-class precision/recall/F1, confusion matrices on the official validation and test splits
- Explainability: SHAP attributions on the linear baseline (words driving a fake prediction)

**Inputs.** Place `multimodal_train.tsv`, `multimodal_validate.tsv`, and `multimodal_test_public.tsv` on Drive (or a Kaggle input path) and set `DATA_DIR` in the configuration cell.


## Environment

Requires a GPU runtime for DistilBERT fine-tuning. CPU is sufficient for EDA, the TF-IDF baseline, and SHAP.

> **Note.** In Colab, set **Runtime → Change runtime type → GPU** (T4 or better). DistilBERT training on CPU is not practical.


## Dependencies

> **Note.** Re-run this cell only after a factory-reset runtime. Colab/Kaggle GPU images already ship a CUDA build of PyTorch.


In [ ]:
# Text baseline + SHAP
%pip install -q scikit-learn pandas numpy matplotlib seaborn shap

# HuggingFace stack for DistilBERT fine-tuning
%pip install -q "transformers>=4.40" datasets accelerate evaluate

# PyTorch with CUDA is provided by Colab/Kaggle GPU images.


> **Note.** After installing packages on a new Colab/Kaggle session, restart the kernel once so `transformers` and `shap` load from the updated environment.


In [ ]:
# --- Standard library / numerics ---
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Classical ML baseline ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

import torch

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


## Dataset path

Fakeddit TSVs are not stored in git (`dataset/` is gitignored). On Colab, mount Drive and set `DATA_DIR` to the folder that contains the three split files. On Kaggle, set `USE_GOOGLE_DRIVE = False` and point `DATA_DIR` at the attached input directory.

> **Note.** Expected files: `multimodal_train.tsv`, `multimodal_validate.tsv`, `multimodal_test_public.tsv`. Check dataset licenses before redistributing trained weights or a public demo.


In [ ]:
# True  → mount Google Drive (Colab)
# False → local / Kaggle path in DATA_DIR
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/fakeddit"
else:
    # Kaggle: "/kaggle/input/<dataset>"  |  session upload: "/content/dataset"
    DATA_DIR = "/content/dataset"

TRAIN_PATH = os.path.join(DATA_DIR, "multimodal_train.tsv")
VAL_PATH = os.path.join(DATA_DIR, "multimodal_validate.tsv")
TEST_PATH = os.path.join(DATA_DIR, "multimodal_test_public.tsv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    assert os.path.exists(p), f"Missing file: {p}"
print("All split files found.")


## Load official splits

Train, validation, and public test come from Fakeddit’s published files. They are not reshuffled.

`SUBSAMPLE_TRAIN` (and later `BERT_TRAIN_MAX`) are compute controls for hosted GPUs. Set them to `None` for full-corpus training used in the evaluation report.

> **Note.** Do not merge splits and call `train_test_split`. Fit vectorizers and tokenizers on train only; a stratified subsample of train is allowed for iteration.


In [ ]:
# Fakeddit 2-way: 0 = fake, 1 = real
LABEL_COL = "2_way_label"
TEXT_COL = "clean_title"

# None = full training split. Integer = stratified subsample for hosted-GPU runs.
SUBSAMPLE_TRAIN = 50_000


def load_split(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", low_memory=False)
    needed = [TEXT_COL, LABEL_COL, "id", "image_url", "hasImage", "subreddit", "title"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")
    return df


# Official Fakeddit splits — do not merge and reshuffle
train_df = load_split(TRAIN_PATH)
val_df = load_split(VAL_PATH)
test_df = load_split(TEST_PATH)

print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

# Optional: stratified subsample of train only (val/test stay full)
if SUBSAMPLE_TRAIN is not None and SUBSAMPLE_TRAIN < len(train_df):
    parts = []
    per_class = SUBSAMPLE_TRAIN // 2
    for label, group in train_df.groupby(LABEL_COL):
        n = min(len(group), per_class)
        parts.append(group.sample(n=n, random_state=SEED))
    train_df = pd.concat(parts, ignore_index=True)
    if len(train_df) > SUBSAMPLE_TRAIN:
        train_df = train_df.sample(n=SUBSAMPLE_TRAIN, random_state=SEED)
    train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    print(f"Stratified train subsample: {len(train_df):,} rows")
    print(train_df[LABEL_COL].value_counts().sort_index())


## Exploratory analysis

Class balance (2-way / 3-way), title-length distribution, missing `image_url` rates, and subreddit mix. Image-url gaps are recorded here and filtered in the vision notebook; this branch uses text only.

> **Note.** The 2-way train split is skewed (~61% fake / ~39% real). Report per-class precision, recall, and F1 — not accuracy alone.


In [ ]:
def label_summary(df: pd.DataFrame, name: str) -> None:
    print(f"\n=== {name} ===")
    print("2_way_label value_counts:")
    print(df[LABEL_COL].value_counts(dropna=False).sort_index())
    if "3_way_label" in df.columns:
        print("\n3_way_label:")
        print(df["3_way_label"].value_counts(dropna=False).sort_index())
    # Recorded for the vision branch; unused as a filter here
    null_img = df["image_url"].isna().sum() + (df["image_url"].astype(str).str.strip() == "").sum()
    print(f"null/empty image_url: {null_img}")
    print("top subreddits:")
    print(df["subreddit"].value_counts().head(5))


for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    label_summary(df, name)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: 2-way class counts (expect fake majority)
train_df[LABEL_COL].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#ff7a59", "#4cd3c2"]
)
axes[0].set_title("Train 2-way label counts (0=fake, 1=real)")
axes[0].set_xlabel("label")
axes[0].set_ylabel("count")

# Right: title length (clip long tail for readability)
lengths = train_df[TEXT_COL].fillna("").astype(str).str.len()
axes[1].hist(lengths.clip(upper=lengths.quantile(0.99)), bins=40, color="#4cd3c2", edgecolor="none")
axes[1].set_title("clean_title length (clipped at 99th pct)")
axes[1].set_xlabel("characters")

plt.tight_layout()
plt.show()

print(lengths.describe())


## Preprocessing

Row-local normalization: lowercase, strip URLs and HTML, collapse whitespace. Punctuation and stylistic markers are kept — they can be predictive for this task.

Vectorizers and tokenizers are fit on **train only**. The same `clean_text` function is applied to validation and test.

> **Note.** Over-cleaning (dropping punctuation, emojis, or clickbait phrasing) erases style signal that often distinguishes fake news. Inference must use this same cleaner.


In [ ]:
# Row-local cleaner — no corpus statistics, safe to apply to all splits
URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"\s+")


def clean_text(text) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = WS_RE.sub(" ", text).strip()
    return text


def prepare_xy(df: pd.DataFrame):
    out = df.copy()
    out["text"] = out[TEXT_COL].map(clean_text)
    out = out[out["text"].str.len() > 0].copy()
    X = out["text"].tolist()
    y = out[LABEL_COL].astype(int).to_numpy()
    return out, X, y


train_prep, X_train, y_train = prepare_xy(train_df)
val_prep, X_val, y_val = prepare_xy(val_df)
test_prep, X_test, y_test = prepare_xy(test_df)

print(len(X_train), len(X_val), len(X_test))
print("Example:", X_train[0], "→", y_train[0])


## Baseline — TF-IDF + logistic regression

Unigram/bigram TF-IDF with class-weighted logistic regression. This is the linear reference the transformer must beat, and the model used for SHAP (exact attributions on a linear pipeline).

> **Note.** `class_weight="balanced"` compensates for the fake/real skew. The DistilBERT run below is the primary text encoder; this baseline is kept for comparison and explanations.


In [ ]:
# Fit TF-IDF on train only (vectorizer lives inside the pipeline)
baseline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=50_000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=200,
                class_weight="balanced",  # ~61/39 fake/real skew
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    ]
)

baseline.fit(X_train, y_train)
print("TF-IDF + LogReg fitted.")


In [ ]:
LABEL_NAMES = ["fake", "real"]


def evaluate_classifier(name: str, y_true, y_pred) -> dict:
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")
    print(f"\n===== {name} =====")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 macro: {f1_macro:.4f} | F1 weighted: {f1_weighted:.4f}")
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3.5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=LABEL_NAMES,
        yticklabels=LABEL_NAMES,
        ax=ax,
    )
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(name)
    plt.tight_layout()
    plt.show()
    return {"accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted}


val_pred_base = baseline.predict(X_val)
test_pred_base = baseline.predict(X_test)

baseline_val_metrics = evaluate_classifier("TF-IDF+LogReg — VAL", y_val, val_pred_base)
baseline_test_metrics = evaluate_classifier("TF-IDF+LogReg — TEST", y_test, test_pred_base)


## Text encoder — DistilBERT

Fine-tune `distilbert-base-uncased` for binary classification. Titles are short, so `MAX_LEN=64`. Checkpoints are written under `DATA_DIR/checkpoints/module01_distilbert` for the fusion stage.

> **Note.** Move both model and batches to GPU (`DEVICE`). If training OOMs, lower `BATCH_SIZE` or `BERT_TRAIN_MAX`. Use the same tokenizer at inference as at training.


In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)
import evaluate as hf_evaluate
import inspect

set_seed(SEED)

# --- Hyperparameters ---
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 64
EPOCHS = 2
BATCH_SIZE = 32 if DEVICE == "cuda" else 8
LR = 2e-5

# None = use the loaded train split. Integer = cap for hosted-GPU memory/time.
BERT_TRAIN_MAX = 30_000

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_hf_dataset(texts, labels):
    return Dataset.from_dict({"text": texts, "labels": list(map(int, labels))})


bert_train_texts, bert_train_labels = X_train, y_train
if BERT_TRAIN_MAX is not None and len(bert_train_texts) > BERT_TRAIN_MAX:
    rng = np.random.RandomState(SEED)
    idx = rng.choice(len(bert_train_texts), size=BERT_TRAIN_MAX, replace=False)
    bert_train_texts = [bert_train_texts[i] for i in idx]
    bert_train_labels = y_train[idx]
    print(f"DistilBERT train size: {len(bert_train_texts):,}")

hf_train = to_hf_dataset(bert_train_texts, bert_train_labels)
hf_val = to_hf_dataset(X_val, y_val)
hf_test = to_hf_dataset(X_test, y_test)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)


# Tokenize with the same tokenizer used at inference
hf_train = hf_train.map(tokenize_batch, batched=True)
hf_val = hf_val.map(tokenize_batch, batched=True)
hf_test = hf_test.map(tokenize_batch, batched=True)

cols = ["input_ids", "attention_mask", "labels"]
hf_train.set_format(type="torch", columns=cols)
hf_val.set_format(type="torch", columns=cols)
hf_test.set_format(type="torch", columns=cols)

# Classification head on DistilBERT; move to GPU when available
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

accuracy_metric = hf_evaluate.load("accuracy")
f1_metric = hf_evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


# Select checkpoint by validation macro-F1
OUTPUT_DIR = "/content/distilbert_fakeddit"
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    seed=SEED,
    report_to="none",
    fp16=(DEVICE == "cuda"),
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Newer transformers: processing_class= ; older: tokenizer=
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_params = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

train_result = trainer.train()
print(train_result)


In [ ]:
# Held-out evaluation on official val / test
val_out = trainer.predict(hf_val)
test_out = trainer.predict(hf_test)

val_pred_bert = np.argmax(val_out.predictions, axis=-1)
test_pred_bert = np.argmax(test_out.predictions, axis=-1)

bert_val_metrics = evaluate_classifier("DistilBERT — VAL", y_val, val_pred_bert)
bert_test_metrics = evaluate_classifier("DistilBERT — TEST", y_test, test_pred_bert)

# Persist weights + tokenizer for fusion / demo
SAVE_DIR = os.path.join(DATA_DIR, "checkpoints", "module01_distilbert")
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved DistilBERT checkpoint to: {SAVE_DIR}")


## Explainability — SHAP (text)

Linear SHAP on the TF-IDF classifier yields per-token contributions. For a held-out example, the five features with the largest contribution toward **fake** are reported. Faithfulness (deletion tests) is evaluated in a later notebook.

> **Note.** A plausible heatmap is not enough. Later evaluation masks these top tokens and checks that model confidence actually drops (faithfulness).


In [ ]:
import shap

vectorizer = baseline.named_steps["tfidf"]
clf = baseline.named_steps["clf"]

# Background sample for LinearExplainer
bg_size = min(200, len(X_train))
background = vectorizer.transform(X_train[:bg_size])
explainer = shap.LinearExplainer(clf, background, feature_dependence="independent")

# One held-out example labeled fake
fake_candidates = [i for i, y in enumerate(y_test) if y == 0]
example_idx = fake_candidates[0] if fake_candidates else 0
example_text = X_test[example_idx]
example_true = int(y_test[example_idx])
example_pred = int(baseline.predict([example_text])[0])
example_proba = baseline.predict_proba([example_text])[0]

print("Example text:", example_text)
print(f"True={LABEL_NAMES[example_true]} | Pred={LABEL_NAMES[example_pred]} | proba={example_proba}")

x_example = vectorizer.transform([example_text])
shap_values = explainer.shap_values(x_example)

# Binary LogReg attributions are typically for class 1 (real). Negate so
# positive values correspond to contribution toward fake (class 0).
if isinstance(shap_values, list):
    sv = np.asarray(shap_values[0][0]).ravel()
else:
    sv = (-np.asarray(shap_values[0])).ravel()

feature_names = np.array(vectorizer.get_feature_names_out())
top_idx = np.argsort(sv)[-5:][::-1]
top_words = [(feature_names[i], float(sv[i])) for i in top_idx]

print("\nTop 5 SHAP features toward fake:")
for rank, (word, score) in enumerate(top_words, start=1):
    print(f"{rank}. {word!r:30s}  shap={score:+.6f}")


In [ ]:
# Feature-importance bar plot for the same example
try:
    shap.plots.bar(
        shap.Explanation(
            values=sv,
            feature_names=feature_names,
            data=x_example.toarray()[0],
        ),
        max_display=10,
        show=True,
    )
except Exception as e:
    print("SHAP bar plot unavailable:", e)


## Results

Validation and test metrics for both models. CSV is written next to the DistilBERT checkpoint. Full-corpus numbers require `SUBSAMPLE_TRAIN = None` and `BERT_TRAIN_MAX = None`.

> **Note.** Subsampled runs are for iteration. Report-quality ablation numbers must be trained on the full official train split.


In [ ]:
# Ablation-table rows: text-only baseline vs DistilBERT
summary = pd.DataFrame(
    [
        {"model": "TF-IDF + LogReg", "split": "test", **baseline_test_metrics},
        {"model": "DistilBERT", "split": "test", **bert_test_metrics},
        {"model": "TF-IDF + LogReg", "split": "val", **baseline_val_metrics},
        {"model": "DistilBERT", "split": "val", **bert_val_metrics},
    ]
)
display(summary.sort_values(["split", "model"]))

summary_path = os.path.join(DATA_DIR, "checkpoints", "module01_metrics.csv")
os.makedirs(os.path.dirname(summary_path), exist_ok=True)
summary.to_csv(summary_path, index=False)
print(f"Wrote {summary_path}")


## Artifacts

| Output | Location |
|--------|----------|
| DistilBERT weights + tokenizer | `{DATA_DIR}/checkpoints/module01_distilbert/` |
| Metrics table | `{DATA_DIR}/checkpoints/module01_metrics.csv` |

These scores are the **text-only** entries in the project ablation table. The next notebook trains the image encoder (ResNet50) and Grad-CAM on the same splits.

> **Note.** Load checkpoints once at app startup in the demo — not on every prediction. Keep train and inference preprocessing identical.

Memory: lower `max_features` or `SUBSAMPLE_TRAIN` if TF-IDF exhausts RAM; lower `BATCH_SIZE` or `BERT_TRAIN_MAX` if DistilBERT OOMs.
